<a href="https://colab.research.google.com/github/Arddiand761/Arddiand761/blob/main/SMOTE_Ipf_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -U imbalanced-learn

In [13]:
from google.colab import drive

# mounting dataset dari gdrive
drive.mount('/content/gdrive', force_remount=True)

# lokasi dataset - ubah sesuai dengan lokasi anda mengupload folder datanya
root_path = 'gdrive/My Drive/Colab Notebooks/Deep Learning Labs/Framingham/'

# opsional - tampilkan info lokasi dataset
print("Path root:", root_path)

Mounted at /content/gdrive
Path root: gdrive/My Drive/Colab Notebooks/Deep Learning Labs/Framingham/


In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings("ignore")


In [15]:
df = pd.read_csv(root_path + "framingham.csv")
print("Jumlah missing value per kolom:\n", df.isnull().sum())


Jumlah missing value per kolom:
 male                 0
age                  0
education          105
currentSmoker        0
cigsPerDay          29
BPMeds              53
prevalentStroke      0
prevalentHyp         0
diabetes             0
totChol             50
sysBP                0
diaBP                0
BMI                 19
heartRate            1
glucose            388
TenYearCHD           0
dtype: int64


In [17]:
df = df.dropna()


In [18]:
X = df.drop("TenYearCHD", axis=1)
y = df["TenYearCHD"]


In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=True)


In [20]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [21]:
svm = SVC()
svm.fit(X_train_scaled, y_train)
y_pred = svm.predict(X_test_scaled)

print("Classification Report Tanpa SMOTE-IPF:\n", classification_report(y_test, y_pred))
print("F1-Score:", f1_score(y_test, y_pred))


Classification Report Tanpa SMOTE-IPF:
               precision    recall  f1-score   support

           0       0.84      1.00      0.91       920
           1       0.00      0.00      0.00       178

    accuracy                           0.84      1098
   macro avg       0.42      0.50      0.46      1098
weighted avg       0.70      0.84      0.76      1098

F1-Score: 0.0


In [22]:
cv = StratifiedKFold(n_splits=5)
pipeline_svm = Pipeline([("scaler", StandardScaler()), ("svm", SVC())])
scores = cross_val_score(pipeline_svm, X, y, cv=cv, scoring="f1")
print("Rata-rata F1-Score CV (tanpa SMOTE-IPF):", scores.mean())


Rata-rata F1-Score CV (tanpa SMOTE-IPF): 0.01692363902082572


In [28]:
from sklearn.neighbors import KNeighborsClassifier
from imblearn.over_sampling import SMOTE
import numpy as np

class SMOTE_IPF:
    def __init__(self, smote_k=5, ipf_k=3, max_iter=5):
        self.smote_k = smote_k
        self.ipf_k = ipf_k
        self.max_iter = max_iter
        self.smote = SMOTE(k_neighbors=self.smote_k)

    def fit_resample(self, X, y):
        # Step 1: SMOTE
        X_resampled, y_resampled = self.smote.fit_resample(X, y)

        # Step 2: Iterative Partitioning Filter (IPF)
        y_array = np.array(y_resampled).reshape(-1, 1)
        data = np.hstack((X_resampled, y_array))

        for _ in range(self.max_iter):
            clf = KNeighborsClassifier(n_neighbors=self.ipf_k)
            X_curr = data[:, :-1]
            y_curr = data[:, -1]

            clf.fit(X_curr, y_curr)
            y_pred = clf.predict(X_curr)

            # Find misclassified points (noisy samples)
            misclassified = y_pred != y_curr

            # If no misclassified, break
            if not np.any(misclassified):
                break

            # Remove misclassified samples
            data = data[~misclassified]

        return data[:, :-1], data[:, -1].astype(int)

In [29]:
smote_ipf = SMOTE_IPF(smote_k=5, ipf_k=3, max_iter=5)
X_resampled, y_resampled = smote_ipf.fit_resample(X_train_scaled, y_train)


In [30]:
svm_resampled = SVC()
svm_resampled.fit(X_resampled, y_resampled)
y_pred_resampled = svm_resampled.predict(X_test_scaled)

print("Classification Report dengan SMOTE-IPF:\n", classification_report(y_test, y_pred_resampled))
print("F1-Score:", f1_score(y_test, y_pred_resampled))


Classification Report dengan SMOTE-IPF:
               precision    recall  f1-score   support

           0       0.89      0.62      0.73       920
           1       0.24      0.61      0.34       178

    accuracy                           0.62      1098
   macro avg       0.56      0.62      0.54      1098
weighted avg       0.79      0.62      0.67      1098

F1-Score: 0.34222919937205654


In [31]:
from imblearn.pipeline import make_pipeline as make_pipeline_imb

pipeline_resampled = make_pipeline_imb(StandardScaler(), SMOTE_IPF(), SVC())
scores_resampled = cross_val_score(pipeline_resampled, X, y, cv=cv, scoring="f1")
print("Rata-rata F1-Score CV (dengan SMOTE-IPF):", scores_resampled.mean())


Rata-rata F1-Score CV (dengan SMOTE-IPF): 0.33387999267471413
